# Assignment 1 — DataCollectionAgent EDA

Этот ноутбук показывает учебный сценарий для задания 1:

- запуск `DataCollectionAgent`
- получение унифицированного датасета
- базовый EDA по текстовой коллекции
- краткий вывод по источникам

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "agents").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from agents.data_collection_agent import DataCollectionAgent

agent = DataCollectionAgent(config=ROOT / "config.yaml")
raw_csv = ROOT / "data/raw/merged_raw.csv"

df = pd.read_csv(raw_csv) if raw_csv.exists() else agent.run()
report = agent.eda(df)

print(df.shape)
df.head()

In [ ]:
summary = pd.DataFrame(
    [
        {"metric": "rows", "value": len(df)},
        {"metric": "sources", "value": df["source"].nunique() if "source" in df.columns else 0},
        {"metric": "labels", "value": df["label"].nunique(dropna=True) if "label" in df.columns else 0},
        {"metric": "mean_text_len_chars", "value": round(report["text_length_chars"]["mean"], 2)},
        {"metric": "mean_text_len_words", "value": round(report["text_length_words"]["mean"], 2)},
    ]
)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pd.Series(report["source_distribution"]).sort_values(ascending=False).plot(kind="bar", ax=axes[0], color="slateblue")
axes[0].set_title("Source distribution")
axes[0].set_ylabel("count")

pd.Series(report["class_distribution"]).sort_values(ascending=False).plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("Class distribution")
axes[1].set_ylabel("count")

plt.tight_layout()
plt.show()

pd.DataFrame(report.get("top20_words", []))

## Краткий вывод

В этой ячейке удобно зафиксировать выводы для защиты:

- какие источники доминируют;
- какие источники лучше подходят как `native_ru_seed` или `unsafe_donor_ru`;
- есть ли перекос по классам;
- достаточно ли тексты похожи на будущие prompt-like примеры.